# Minimal LoRA XLM-R + Tuned Rounding

One-pass baseline using the same standard `lora_bert` setup as `minimal_locc_lora_xlmr`, but with only one training epoch. After training, validation predictions are used to learn monotone rounding thresholds that minimize validation MAE.

Naive rounding uses fixed boundaries `0.5, 1.5, 2.5, 3.5`. Tuned rounding learns `t1 < t2 < t3 < t4`, which can compensate for class imbalance, label noise, default-0 artifacts, and regression bias.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import inspect
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, Value
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, Trainer, TrainingArguments, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from experiments.config import ModelConfig
from experiments.models import SentimentModel

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_tuned_rounding_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128

EPOCHS = 1
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1024
LR = 1.5e-4
FP16 = torch.cuda.is_available()

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data and tokenization

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")
df["lang"] = df.get("lang", "unk")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_hard(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [float(x) for x in batch["label"]]
    langs = batch["lang"] if "lang" in batch else ["unk"] * len(batch["sentence"])
    out["lang"] = [0 if x == "eng_Latn" else 1 for x in langs]
    return out

def to_hf_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize_hard, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("labels", Value("float32"))
    ds = ds.cast_column("lang", Value("int64"))
    ds.set_format("torch")
    return ds

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)

## Model and trainer

In [ ]:
def make_model():
    cfg = ModelConfig(
        kind="lora_bert",
        name=MODEL_ID,
        geometry="default",
        lora_r=128,
        lora_alpha=64,
        lora_dropout=0.01,
    )
    model = SentimentModel.from_config(cfg)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


def apply_thresholds(scores, thresholds):
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    thresholds = np.asarray(thresholds, dtype=np.float32)
    return np.searchsorted(thresholds, scores, side="right").astype(int)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.asarray(logits).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    rounded = np.rint(np.clip(preds, 0, 4))
    return {
        "mae": float(mean_absolute_error(labels, preds)),
        "rounded_mae": float(mean_absolute_error(labels, rounded)),
    }


def make_training_args(run_name):
    kwargs = dict(
        output_dir=str(OUTPUT_DIR / run_name / "checkpoints"),
        overwrite_output_dir=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=EPOCHS,
        evaluation_strategy="steps",
        eval_steps=500,
        logging_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        seed=SEED,
    )
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)

## Train for 1 epoch

In [ ]:
model = make_model()
trainer = Trainer(
    model=model,
    args=make_training_args("base_1epoch"),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
#metrics = trainer.evaluate()
#metrics

## Tune rounding thresholds on validation MAE

This tuner is exact for threshold classifiers on the validation predictions. It groups equal scores, then uses dynamic programming to find the best ordered partition into classes `0..4` under MAE.

In [ ]:
def tune_mae_thresholds(scores, labels, n_classes=5):
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    labels = np.asarray(labels, dtype=int).reshape(-1)
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_labels = labels[order]

    unique_scores, group_starts = np.unique(sorted_scores, return_index=True)
    group_ends = np.r_[group_starts[1:], len(sorted_scores)]
    n_groups = len(unique_scores)

    group_cost = np.zeros((n_classes, n_groups), dtype=np.float64)
    for g, (start, end) in enumerate(zip(group_starts, group_ends)):
        y = sorted_labels[start:end]
        for cls in range(n_classes):
            group_cost[cls, g] = np.abs(cls - y).sum()

    prefix_cost = np.c_[np.zeros(n_classes), np.cumsum(group_cost, axis=1)]
    dp = np.full((n_classes, n_groups + 1), np.inf, dtype=np.float64)
    back = np.zeros((n_classes, n_groups + 1), dtype=int)
    dp[0] = prefix_cost[0]

    for cls in range(1, n_classes):
        best_value = np.inf
        best_split = 0
        for j in range(n_groups + 1):
            candidate = dp[cls - 1, j] - prefix_cost[cls, j]
            if candidate < best_value:
                best_value = candidate
                best_split = j
            dp[cls, j] = prefix_cost[cls, j] + best_value
            back[cls, j] = best_split

    cuts = []
    j = n_groups
    for cls in range(n_classes - 1, 0, -1):
        j = back[cls, j]
        cuts.append(j)
    cuts = cuts[::-1]

    thresholds = []
    eps = 1e-6
    for cut in cuts:
        if cut <= 0:
            thresholds.append(float(unique_scores[0] - eps))
        elif cut >= n_groups:
            thresholds.append(float(unique_scores[-1] + eps))
        else:
            thresholds.append(float((unique_scores[cut - 1] + unique_scores[cut]) / 2.0))

    tuned_preds = apply_thresholds(scores, thresholds)
    return np.array(thresholds, dtype=np.float32), tuned_preds, float(mean_absolute_error(labels, tuned_preds))

In [ ]:
val_raw = trainer.predict(val_ds).predictions.reshape(-1)
val_labels = val_df["label"].to_numpy(dtype=int)

naive_thresholds = np.array([0.5, 1.5, 2.5, 3.5], dtype=np.float32)
naive_preds = apply_thresholds(np.clip(val_raw, 0, 4), naive_thresholds)
tuned_thresholds, tuned_preds, tuned_mae = tune_mae_thresholds(np.clip(val_raw, 0, 4), val_labels)

print("raw regression MAE:", mean_absolute_error(val_labels, val_raw))
print("naive thresholds:", naive_thresholds.tolist())
print("naive rounded MAE:", mean_absolute_error(val_labels, naive_preds))
print("tuned thresholds:", tuned_thresholds.tolist())
print("tuned rounded MAE:", tuned_mae)

pd.crosstab(
    pd.Series(val_labels, name="label"),
    pd.Series(tuned_preds, name="tuned_pred"),
    margins=True,
)

In [ ]:
final_dir = OUTPUT_DIR / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
threshold_path = OUTPUT_DIR / "tuned_thresholds.npy"
np.save(threshold_path, tuned_thresholds)
print("model:", final_dir)
print("thresholds:", threshold_path)

## Optional submission with tuned thresholds

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")
    test_raw = trainer.predict(test_ds).predictions.reshape(-1)
    test_preds = apply_thresholds(np.clip(test_raw, 0, 4), tuned_thresholds)
    submission = pd.DataFrame({"id": test_df["id"], "label": test_preds.astype(int)})
    submission_path = OUTPUT_DIR / "submission_tuned_rounding.csv"
    submission.to_csv(submission_path, index=False)
    print(submission_path)
    display(submission.head())